In [0]:
from pyspark.sql.functions import * 

spark.conf.set(
"fs.azure.account.key.ecomdatalakegen2.dfs.core.windows.net",
dbutils.secrets.get(
    scope="azure",
    key="storage-key"
))

storage_account = "ecomdatalakegen2"
bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net"
checkpoint_base = f"{silver_base}/_checkpoints"

clickstream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/clickstream_schema")
    .load(f"{bronze_base}/streaming/")
    .dropDuplicates(["event_id"])
    .withColumn("timestamp", to_timestamp("timestamp"))
)


clickstream_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_base}/clickstream_write") \
    .outputMode("append") \
    .start(f"{silver_base}/clickstream")